In [1]:

# Titanic Survivor Data Preparation Pipeline

# Step 1: Import required libraries

import pandas as pd
import seaborn as sns
import numpy as np

print("Libraries imported successfully!")


Libraries imported successfully!


In [2]:

# Stage 2: Load Titanic Dataset

def load_data():

    data = sns.load_dataset("titanic")

    return data


# Load the dataset
df = load_data()

# Display the dataset
print("Titanic Dataset Loaded Successfully!")

print("Dataset Shape:", df.shape)

display(df.head())


Titanic Dataset Loaded Successfully!
Dataset Shape: (891, 15)


,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True


In [3]:

# Stage 3: Validate Titanic Dataset

# Create a custom exception
class MissingColumnError(Exception):
    pass


# Function to validate required columns
def validate_data(data):

    required_columns = [
        "survived",
        "pclass",
        "sex",
        "age",
        "fare",
        "embarked"
    ]

    missing_columns = [
        col for col in required_columns
        if col not in data.columns
    ]

    if missing_columns:
        raise MissingColumnError(
            f"Missing required columns: {missing_columns}"
        )

    print("Data validation successful!")

    return data


# Validate Titanic dataset
df = validate_data(df)

print("Dataset is ready for cleaning.")


Data validation successful!
Dataset is ready for cleaning.


In [4]:
# Test custom exception

test_df = df.drop(columns=["age"])

try:
    validate_data(test_df)

except MissingColumnError as e:
    print("Validation Error:", e)

Validation Error: Missing required columns: ['age']


In [5]:

# Stage 4: Clean Text

def clean_text(data):

    text_columns = data.select_dtypes(
        include=["object", "category"]
    ).columns

    for col in text_columns:
        data[col] = data[col].apply(
            lambda x: x.strip().lower()
            if isinstance(x, str) else x
        )

    return data


# Clean Titanic dataset
df = clean_text(df)

print("Text cleaning completed successfully!")

display(df.head())


Text cleaning completed successfully!


,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
0,0,3,male,22.0,1,0,7.2500,s,third,man,True,NaN,southampton,no,False
1,1,1,female,38.0,1,0,71.2833,c,first,woman,False,c,cherbourg,yes,False
2,1,3,female,26.0,0,0,7.9250,s,third,woman,False,NaN,southampton,yes,True
3,1,1,female,35.0,1,0,53.1000,s,first,woman,False,c,southampton,yes,False
4,0,3,male,35.0,0,0,8.0500,s,third,man,True,NaN,southampton,no,True


In [6]:

# Stage 5: Drop Useless Columns

def drop_useless(data):

    useless_columns = [
        "alive",
        "class",
        "embark_town",
        "who",
        "adult_male",
        "deck"
    ]

    data = data.drop(
        columns=useless_columns,
        errors="ignore"
    )

    return data


# Remove unnecessary columns
df = drop_useless(df)

print("Unnecessary columns removed successfully!")

print("Remaining columns:", df.columns.tolist())

print("Dataset shape:", df.shape)


Unnecessary columns removed successfully!
Remaining columns: ['survived', 'pclass', 'sex', 'age', 'sibsp', 'parch', 'fare', 'embarked', 'alone']
Dataset shape: (891, 9)


In [7]:

# Stage 6: Handle Missing Values

def handle_missing(data):

    # Separate numerical and categorical columns
    numeric_columns = data.select_dtypes(
        include=["number"]
    ).columns

    categorical_columns = data.select_dtypes(
        include=["object", "category"]
    ).columns

    # Fill numerical missing values with median
    for col in numeric_columns:
        data[col] = data[col].fillna(
            data[col].median()
        )

    # Fill categorical missing values with mode
    for col in categorical_columns:
        data[col] = data[col].fillna(
            data[col].mode()[0]
        )

    return data


# Handle missing values
df = handle_missing(df)

print("Missing values handled successfully!")

print("\nRemaining missing values:")
print(df.isnull().sum())

print("\nTotal missing values:", df.isnull().sum().sum())


Missing values handled successfully!

Remaining missing values:
survived    0
pclass      0
sex         0
age         0
sibsp       0
parch       0
fare        0
embarked    0
alone       0
dtype: int64

Total missing values: 0


In [8]:

# Stage 7: Encode Categorical Data

def encode_categoricals(data):

    # Binary encoding
    data["sex"] = data["sex"].map({
        "male": 0,
        "female": 1
    })

    # Ordinal encoding
    data["embarked"] = data["embarked"].map({
        "s": 1,
        "c": 2,
        "q": 3
    })

    # One-hot encoding
    data = pd.get_dummies(
        data,
        columns=["embarked"],
        dtype=int
    )

    return data


# Encode categorical columns
df = encode_categoricals(df)

print("Categorical encoding completed successfully!")

display(df.head())

print("Final columns:", df.columns.tolist())


Categorical encoding completed successfully!


,survived,pclass,sex,age,sibsp,parch,fare,alone,embarked_1,embarked_2,embarked_3
0,0,3,0,22.0,1,0,7.2500,False,1,0,0
1,1,1,1,38.0,1,0,71.2833,False,0,1,0
2,1,3,1,26.0,0,0,7.9250,True,1,0,0
3,1,1,1,35.0,1,0,53.1000,False,1,0,0
4,0,3,0,35.0,0,0,8.0500,True,1,0,0


Final columns: ['survived', 'pclass', 'sex', 'age', 'sibsp', 'parch', 'fare', 'alone', 'embarked_1', 'embarked_2', 'embarked_3']


In [9]:

# Stage 8: Split Data

from sklearn.model_selection import train_test_split

def split_data(data):

    # Separate target and features
    X = data.drop(columns=["survived"])
    y = data["survived"]

    # Split into training and testing data
    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.20,
        random_state=42,
        stratify=y
    )

    return X_train, X_test, y_train, y_test


# Split Titanic dataset
X_train, X_test, y_train, y_test = split_data(df)

print("Data splitting completed successfully!")

print("Training features:", X_train.shape)
print("Testing features:", X_test.shape)

print("Training target:", y_train.shape)
print("Testing target:", y_test.shape)


Data splitting completed successfully!
Training features: (712, 10)
Testing features: (179, 10)
Training target: (712,)
Testing target: (179,)


In [10]:

# Stage 9: Scale Features

from sklearn.preprocessing import StandardScaler

def scale_features(X_train, X_test):

    scaler = StandardScaler()

    # Fit only on training data
    X_train_scaled = scaler.fit_transform(X_train)

    # Transform testing data
    X_test_scaled = scaler.transform(X_test)

    return X_train_scaled, X_test_scaled, scaler


# Scale Titanic features
X_train_scaled, X_test_scaled, scaler = scale_features(
    X_train, X_test
)

print("Feature scaling completed successfully!")

print("Scaled training data:", X_train_scaled.shape)
print("Scaled testing data:", X_test_scaled.shape)


Feature scaling completed successfully!
Scaled training data: (712, 10)
Scaled testing data: (179, 10)


In [11]:

# Stage 10: Main Function

def main():

    try:
        print("Starting Titanic Data Preparation Pipeline...")

        # 1. Load dataset
        data = load_data()

        # 2. Validate dataset
        data = validate_data(data)

        # 3. Clean text
        data = clean_text(data)

        # 4. Drop unnecessary columns
        data = drop_useless(data)

        # 5. Handle missing values
        data = handle_missing(data)

        # 6. Encode categorical data
        data = encode_categoricals(data)

        # 7. Split dataset
        X_train, X_test, y_train, y_test = split_data(data)

        # 8. Scale features
        X_train_scaled, X_test_scaled, scaler = scale_features(
            X_train, X_test
        )

        print("\nPIPELINE COMPLETED SUCCESSFULLY!")

        print("Training data:", X_train_scaled.shape)
        print("Testing data:", X_test_scaled.shape)

        return (
            X_train_scaled,
            X_test_scaled,
            y_train,
            y_test,
            scaler
        )

    except Exception as e:
        print("Pipeline Error:", e)


# Run complete pipeline
results = main()


Starting Titanic Data Preparation Pipeline...
Data validation successful!

PIPELINE COMPLETED SUCCESSFULLY!
Training data: (712, 10)
Testing data: (179, 10)


In [12]:
!python Titanic_Survivor_Data_Preparation.py

Starting Titanic Data Preparation Pipeline...
Dataset loaded: (891, 15)
Data validation successful!
After dropping unnecessary columns: (891, 9)
Remaining missing values: 0
After categorical encoding: (891, 11)

PIPELINE COMPLETED SUCCESSFULLY!
Training features: (712, 10)
Testing features: (179, 10)
Training target: (712,)
Testing target: (179,)
